# Lesson 41: Encoder-decoder improvements activity

In this activity, you will experiment with modifications to the encoder-decoder architecture from the lesson demo.

1. **Embedding layer** - Add an embedding layer instead of one-hot encoding
2. **Bidirectional encoder** - Make the encoder bidirectional
3. **Hyperparameter tuning** - Experiment with latent dimension size

## Notebook set-up

### Imports

In [ ]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Bidirectional, Concatenate
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

np.random.seed(315)

## 1. Prepare data

Using the same English-French translation pairs from the demo.

In [ ]:
# English-French translation pairs
pairs = [
    ('hello', 'bonjour'),
    ('goodbye', 'au revoir'),
    ('thank you', 'merci'),
    ('yes', 'oui'),
    ('no', 'non'),
    ('please', 's il vous plait'),
    ('good morning', 'bonjour'),
    ('good night', 'bonne nuit'),
    ('how are you', 'comment allez vous'),
    ('i am fine', 'je vais bien'),
    ('what is your name', 'comment vous appelez vous'),
    ('my name is', 'je m appelle'),
    ('nice to meet you', 'enchanté'),
    ('see you later', 'à bientôt'),
    ('i love you', 'je t aime'),
]

# Prepare input/output texts with start/end tokens
input_texts = [pair[0] for pair in pairs]
target_texts = ['<start> ' + pair[1] + ' <end>' for pair in pairs]

# Tokenize
input_tokenizer = Tokenizer(filters='')
input_tokenizer.fit_on_texts(input_texts)
target_tokenizer = Tokenizer(filters='')
target_tokenizer.fit_on_texts(target_texts)

# Convert to sequences
encoder_input_seq = input_tokenizer.texts_to_sequences(input_texts)
decoder_input_seq = target_tokenizer.texts_to_sequences(target_texts)

# Pad sequences
max_encoder_len = max(len(seq) for seq in encoder_input_seq)
max_decoder_len = max(len(seq) for seq in decoder_input_seq)

encoder_input_data = pad_sequences(encoder_input_seq, maxlen=max_encoder_len, padding='post')
decoder_input_data = pad_sequences(decoder_input_seq, maxlen=max_decoder_len, padding='post')

# Decoder target (shifted by one position)
decoder_target_data = np.zeros((len(pairs), max_decoder_len, len(target_tokenizer.word_index) + 1))
for i, seq in enumerate(decoder_input_seq):
    for t, word_idx in enumerate(seq[1:], start=0):  # Skip <start>
        decoder_target_data[i, t, word_idx] = 1.0

num_encoder_tokens = len(input_tokenizer.word_index) + 1
num_decoder_tokens = len(target_tokenizer.word_index) + 1

print(f'Encoder vocabulary size: {num_encoder_tokens}')
print(f'Decoder vocabulary size: {num_decoder_tokens}')
print(f'Max encoder sequence length: {max_encoder_len}')
print(f'Max decoder sequence length: {max_decoder_len}')

## 2. Baseline model (from demo)

This is the basic encoder-decoder from the lesson.

In [ ]:
def build_baseline_model(latent_dim=64):
    '''Build basic encoder-decoder with embedding layers.'''

    # Encoder
    encoder_inputs = Input(shape=(max_encoder_len,))
    encoder_embedding = Embedding(num_encoder_tokens, latent_dim)(encoder_inputs)
    encoder_lstm = LSTM(latent_dim, return_state=True)
    _, state_h, state_c = encoder_lstm(encoder_embedding)
    encoder_states = [state_h, state_c]
    
    # Decoder
    decoder_inputs = Input(shape=(max_decoder_len,))
    decoder_embedding = Embedding(num_decoder_tokens, latent_dim)(decoder_inputs)
    decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
    decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
    decoder_dense = Dense(num_decoder_tokens, activation='softmax')
    decoder_outputs = decoder_dense(decoder_outputs)
    
    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    return model

baseline_model = build_baseline_model()
print(f'Baseline model parameters: {baseline_model.count_params():,}')

## 3. Improved model with bidirectional encoder

### Task 1: Add bidirectional encoder

Modify the baseline model to use a bidirectional LSTM encoder. This allows the model to capture context from both directions.

**Hints:**
- Wrap the encoder LSTM with `Bidirectional`
- Use `merge_mode='concat'` (default) to concatenate forward and backward states
- The decoder latent dimension should be `2 * latent_dim` to match the concatenated states
- Use `Concatenate()` to combine forward and backward states for `state_h` and `state_c`

In [ ]:
def build_bidirectional_model(latent_dim=64):
    '''Build encoder-decoder with bidirectional encoder.
    
    TODO: Implement the bidirectional encoder modifications.
    '''

    # Encoder
    encoder_inputs = Input(shape=(max_encoder_len,))
    encoder_embedding = Embedding(num_encoder_tokens, latent_dim)(encoder_inputs)
    
    # TODO: Replace LSTM with Bidirectional LSTM
    # Hint: encoder_lstm = Bidirectional(LSTM(latent_dim, return_state=True))
    encoder_lstm = None  # YOUR CODE HERE
    
    # TODO: Get outputs and states from bidirectional LSTM
    # Hint: outputs, fwd_h, fwd_c, bwd_h, bwd_c = encoder_lstm(encoder_embedding)
    # YOUR CODE HERE
    
    # TODO: Concatenate forward and backward states
    # Hint: state_h = Concatenate()([fwd_h, bwd_h])
    # Hint: state_c = Concatenate()([fwd_c, bwd_c])
    encoder_states = None  # YOUR CODE HERE - should be [state_h, state_c]
    
    # Decoder (note: latent_dim * 2 because of bidirectional concatenation)
    decoder_inputs = Input(shape=(max_decoder_len,))
    decoder_embedding = Embedding(num_decoder_tokens, latent_dim)(decoder_inputs)
    decoder_lstm = LSTM(latent_dim * 2, return_sequences=True, return_state=True)
    decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
    decoder_dense = Dense(num_decoder_tokens, activation='softmax')
    decoder_outputs = decoder_dense(decoder_outputs)
    
    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    return model

## 4. Train and compare models

### Task 2: Compare model performance

Train both models and compare their training accuracy.

In [ ]:
# Train baseline model
print('Training baseline model...')
baseline_model = build_baseline_model(latent_dim=64)
baseline_history = baseline_model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=4,
    epochs=100,
    verbose=0
)
print(f'Baseline final accuracy: {baseline_history.history["accuracy"][-1]:.4f}')

In [ ]:
# TODO: Train bidirectional model (after completing Task 1)
print('Training bidirectional model...')
bidirectional_model = build_bidirectional_model(latent_dim=64)
bidirectional_history = bidirectional_model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=4,
    epochs=100,
    verbose=0
)
print(f'Bidirectional final accuracy: {bidirectional_history.history["accuracy"][-1]:.4f}')

## 5. Hyperparameter experiment

### Task 3: Experiment with latent dimension

Try different values of `latent_dim` (e.g., 32, 64, 128, 256) and observe the effect on training.

In [ ]:
# TODO: Experiment with different latent dimensions
latent_dims = [32, 64, 128]  # Add more values to test
results = {}

for dim in latent_dims:
    print(f'Training with latent_dim={dim}...')
    model = build_baseline_model(latent_dim=dim)
    history = model.fit(
        [encoder_input_data, decoder_input_data],
        decoder_target_data,
        batch_size=4,
        epochs=100,
        verbose=0
    )
    results[dim] = {
        'accuracy': history.history['accuracy'][-1],
        'params': model.count_params()
    }
    print(f'  Accuracy: {results[dim]["accuracy"]:.4f}, Parameters: {results[dim]["params"]:,}')

print('\nSummary:')
for dim, res in results.items():
    print(f'latent_dim={dim:3d}: accuracy={res["accuracy"]:.4f}, params={res["params"]:,}')

## 6. Analysis questions

After completing the tasks above, answer these questions:

1. How does the bidirectional encoder affect model performance? Why might this help for translation?
2. What is the relationship between latent dimension and training accuracy?
3. How does model size (number of parameters) scale with latent dimension?
4. What other modifications could improve translation quality?